# S08: why is |s| > 3 routine? A diagnostic of the S05 s-score (no trading)

Pre-registration: [`S08_README.md`](S08_README.md). Engine: `src/equity_trading/s08.py`.

| Variant | s-score |
|---|---|
| V0 | S05 as traded |
| V1 | overnight bar excluded from the residual (window fit and live) |
| V2 | V1 scaled by a HAR-style volatility ratio (1-, 5-, 20-session realized variance) |
| V3 | V0 scaled by the same volatility ratio |

Target for every variant: the forward sum of S05 residual increments (V0 parameters, overnight included), i.e. the residual return a position actually earns.

In [ ]:
SAMPLE_EVERY = 4             # bars between sampled cross-sections for IC and bucket tables
HORIZONS = {"1h": 4, "1d": 26, "5d": 130}
ROUND_TRIP_BPS = 13.0        # S05b measured cost per trade (stock + hedge, market orders)

import json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, s04, s05, s08
from equity_trading.alpaca import ALPACA_VERSION, download_years

SYNTHETIC = os.environ.get("S08_SYNTHETIC") == "1"
YEARS = list(range(2020, 2026))                      # 2026 not used
CFG = s08.S08Config()
OUT = ROOT / "results" / ("s08_synthetic" if SYNTHETIC else "s08")
OUT.mkdir(parents=True, exist_ok=True)
VARS = ["s0", "s1", "s2", "s3"]
NAMES = {"s0": "V0 S05", "s1": "V1 no overnight", "s2": "V2 no overnight + vol", "s3": "V3 S05 + vol"}
COL = {"s0": "#8a8a84", "s1": "#2a78d6", "s2": "#1baf7a", "s3": "#eb6834"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})
if SYNTHETIC:
    from equity_trading.universe import SECTOR_ETF_01, UNIVERSE_01
    ETF_OF = {s: SECTOR_ETF_01[k] for k, v in UNIVERSE_01.items() for s in v}
    BENCH, YEARS, MIN_N = "XLK", [2024, 2025], 20
else:
    constituents = pd.read_csv(ROOT / "reference" / "sp500_constituents_20260913.csv")
    ETF_OF = dict(zip(constituents["Symbol"], constituents["GICS Sector"].map(s05.SECTOR_ETF)))
    BENCH, MIN_N = "SPY", 50
STOCKS = sorted(ETF_OF)
ETFS = sorted(set(ETF_OF.values()))
print(f"synthetic={SYNTHETIC} | stocks={len(STOCKS)} | {CFG}")

## 1. Data and s-score paths

In [ ]:
t0 = time.time()
if SYNTHETIC:
    panel, _ = s04.synthetic_ohlc_panel(YEARS)
else:
    download_years(STOCKS + ETFS + [BENCH], "15Min", YEARS, workers=8)
    panel = s04.cached_panel(STOCKS + ETFS + [BENCH], "15Min", YEARS, ffill_bars=2)
have = set(panel.symbols)
TRADED = [s for s in STOCKS if s in have and ETF_OF[s] in have]
SC = np.array([panel.symbols.index(s) for s in TRADED])
EC = np.array([panel.symbols.index(ETF_OF[s]) for s in TRADED])
R = s08.compute(panel, SC, EC, CFG)
days = pd.to_datetime(panel.days)
first_live = panel.day_start[CFG.window_sessions]
ROWS = np.arange(first_live, len(panel.index) - max(HORIZONS.values()) - 1, SAMPLE_EVERY)
FWD = {h: s08.forward_sum(R["eps"], n) for h, n in HORIZONS.items()}
print(f"panel {panel.c.shape}, stocks {len(TRADED)}, computed in {(time.time() - t0) / 60:.1f} min; sampled cross-sections {len(ROWS):,}")

## 2. How extreme is s? Exceedance vs a normal OU, and by time of day

In [ ]:
exc = pd.DataFrame({NAMES[v]: s08.exceedance(R[v]) for v in VARS})
exc["normal OU"] = pd.Series(s08.normal_exceedance())
display(exc.T.round(4))
slot = R["slot"]
by_slot = pd.DataFrame({NAMES[v]: pd.Series(np.nanmean(np.where(np.isfinite(R[v]), np.abs(R[v]) > 3, np.nan), axis=1)).groupby(slot).mean()
                        for v in VARS})
fig, ax = plt.subplots(figsize=(9, 3))
for v in VARS:
    ax.plot(by_slot.index, by_slot[NAMES[v]], marker="o", ms=3, color=COL[v], label=NAMES[v])
ax.axhline(0.0027, color="k", lw=0.8, ls=":", label="normal OU")
ax.set_xlabel("bar of session (0 = 09:30)"); ax.set_title("Share of stocks with |s| > 3 at each bar close"); ax.legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()

## 3. Is it the mean? Share of the extreme V0 scores explained by −m/σ

In [ ]:
ms = np.abs(R["m_over_sig0"])
big = np.abs(R["s0"]) > 3
decomp = {"share |m/sigma| > 1": float(np.nanmean(ms > 1)), "share |m/sigma| > 2": float(np.nanmean(ms > 2)),
          "share |m/sigma| > 3": float(np.nanmean(ms > 3)),
          "among |s0| > 3: share with |m/sigma| > 2": float(np.nanmean((ms > 2)[big])),
          "among |s0| > 3: share where sign(-m) = sign(s)": float(np.nanmean((np.sign(R["m_over_sig0"]) == np.sign(R["s0"]))[big]))}
display(pd.Series(decomp).round(4).to_frame("V0"))

## 4. Tails of the OU innovations (is a Student-t warranted?)

In [ ]:
from scipy import stats
kurt = {"V0 (with overnight) median excess kurtosis": float(np.nanmedian(R["kurt0"])),
        "V1 (no overnight) median excess kurtosis": float(np.nanmedian(R["kurt1"]))}
rng = np.random.default_rng(0)
zs = [z for _, _, z in R["z_samples"]]
pick = rng.choice(len(zs), size=min(800, len(zs)), replace=False)
nus = []
for k in pick:
    z = zs[k][np.isfinite(zs[k])]
    try:
        nu, _, _ = stats.t.fit(z, floc=0.0)
        nus.append(min(nu, 200.0))
    except Exception:
        pass
kurt.update({"V1 innovations: Student-t nu, median (sample)": float(np.median(nus)),
             "V1 innovations: share of stock-days with nu < 5": float(np.mean(np.array(nus) < 5))})
display(pd.Series(kurt).round(3).to_frame())

## 5. Signal quality: rank IC of −s against forward residual return

In [ ]:
ic_rows, ic_series = [], {}
for v in VARS:
    for h in HORIZONS:
        ic = s08.rank_ic(-R[v], FWD[h], ROWS, min_n=MIN_N)
        d = pd.Series(ic, index=days[np.searchsorted(panel.day_start, ROWS, side="right") - 1]).groupby(level=0).mean().dropna()
        ic_series[(v, h)] = d
        lag = {"1h": 1, "1d": 1, "5d": 5}[h]
        import statsmodels.api as sm
        nw = sm.OLS(d.to_numpy(), np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": lag})
        ic_rows.append({"variant": NAMES[v], "horizon": h, "mean_IC": float(d.mean()), "t_NW": float(nw.tvalues[0]),
                        "share_days_IC>0": float((d > 0).mean())})
ic_tab = pd.DataFrame(ic_rows).pivot(index="variant", columns="horizon")
display(ic_tab.round(4))
ic_year = pd.DataFrame({NAMES[v]: ic_series[(v, "1d")].groupby(ic_series[(v, "1d")].index.year).mean() for v in VARS})
display(ic_year.round(4).rename_axis("mean IC, 1d horizon"))

## 6. Expected contrarian residual return by s bucket (bps), against the S05b round trip

In [ ]:
bk = {}
for v in VARS:
    for h in ("1d", "5d"):
        bk[(NAMES[v], h)] = s08.bucket_returns(R[v], FWD[h], ROWS)["contrarian_bps"]
bucket_tab = pd.DataFrame(bk)
display(bucket_tab.round(2))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for i, h in enumerate(("1d", "5d")):
    for v in VARS:
        y = bucket_tab[(NAMES[v], h)]
        ax[i].plot(range(len(y)), y.values, marker="o", ms=3, color=COL[v], label=NAMES[v])
    ax[i].axhline(ROUND_TRIP_BPS, color="k", ls=":", lw=0.8, label="S05b round trip")
    ax[i].set_xticks(range(len(y))); ax[i].set_xticklabels([str(b) for b in y.index], rotation=45, fontsize=7)
    ax[i].set_title(f"Contrarian residual return, {h}"); ax[i].legend(frameon=False, fontsize=7)
plt.tight_layout(); plt.show()

## 7. Calibration within the session: P(s reaches 0 before |s| = 4) vs OU

In [ ]:
cal = {}
sub = ROWS[::5]
for v in ("s0", "s1", "s2", "s3"):
    hd = s08.within_day_hits(R[v], panel.day_start, sub)
    hd["bucket"] = pd.cut(hd.abs_s0, [1.25, 1.5, 2, 3, 4], include_lowest=True)
    g = hd.groupby("bucket", observed=True)
    cal[NAMES[v]] = pd.DataFrame({"n": g.size(), "resolved_share": g.hit_zero.apply(lambda x: x.notna().mean()),
                                  "P_zero_first": g.hit_zero.mean(),
                                  "OU": g.abs_s0.apply(lambda x: s08.ou_hit_zero_prob(x).mean())})
display(pd.concat(cal).round(3))

## 8. Regimes: does mean reversion depend on the market volatility state?

In [ ]:
br = np.log(panel.c[:, panel.symbols.index(BENCH)])
r_b = np.diff(br, prepend=np.nan)
r_b[panel.day_start[:-1]] = np.nan                    # intraday only
rv_day = pd.Series([np.nansum(r_b[a:b] ** 2) for a, b in zip(panel.day_start[:-1], panel.day_start[1:])], index=days)
rv20 = rv_day.rolling(20).mean().shift(1)             # known before the day starts
terc = pd.qcut(rv20, 3, labels=["low vol", "mid vol", "high vol"])
reg = {}
for lab in ["low vol", "mid vol", "high vol"]:
    dd = terc.index[terc == lab]
    di = np.isin(days, dd)
    tau = R["tau0"][di]
    rec = {"days": int(di.sum()), "median_tau_sessions": float(np.nanmedian(tau) / 26)}
    for v in ("s0", "s3"):
        s_ = ic_series[(v, "1d")]
        rec[f"IC_1d {NAMES[v]}"] = float(s_[s_.index.isin(dd)].mean())
    row_days = days[np.searchsorted(panel.day_start, ROWS, side="right") - 1]
    rr = ROWS[np.isin(row_days, dd)]
    b = s08.bucket_returns(R["s0"], FWD["1d"], rr)["contrarian_bps"]
    rec["V0 contrarian bps, |s| 2-3, 1d"] = float(b[[i for i in b.index if i.left in (-3.0, 2.0)]].mean())
    reg[lab] = rec
reg_tab = pd.DataFrame(reg).T
display(reg_tab.round(3))

## 9. Pre-stated readings and summary

In [ ]:
def ic(v, h): return float(ic_tab.loc[NAMES[v], ("mean_IC", h)])
best_bps_5d = float(bucket_tab.xs("5d", axis=1, level=1).max().max())
readings = {
    "R1 vol scaling worth porting (V3 IC >= 1.10 x V0 at 1d and 5d)": ic("s3", "1d") >= 1.10 * ic("s0", "1d") and ic("s3", "5d") >= 1.10 * ic("s0", "5d"),
    "R2 overnight exclusion worth adopting (V1 IC >= 1.10 x V0 at 1d and 5d)": ic("s1", "1d") >= 1.10 * ic("s0", "1d") and ic("s1", "5d") >= 1.10 * ic("s0", "5d"),
    "R3 regime model justified (V0 1d IC high vs low vol differs >= 25%)": abs(reg_tab.loc["high vol", "IC_1d V0 S05"] - reg_tab.loc["low vol", "IC_1d V0 S05"])
        >= 0.25 * max(abs(reg_tab.loc["low vol", "IC_1d V0 S05"]), 1e-9),
    "R4 Student-t likelihood worth testing (V1 innovations median excess kurtosis > 3)": kurt["V1 (no overnight) median excess kurtosis"] > 3,
    "R5 mean noise drives extremes (among |s0| > 3, share |m/sigma| > 2 above 50%)": decomp["among |s0| > 3: share with |m/sigma| > 2"] > 0.5,
    "R6 any bucket's 5d contrarian return exceeds the S05b round trip": best_bps_5d > ROUND_TRIP_BPS,
}
display(pd.Series(readings).to_frame("reading"))
summary = {"study": "S08", "synthetic": SYNTHETIC, "years": YEARS, "config": CFG.as_dict(), "exceedance": exc.to_dict(),
           "decomposition": decomp, "innovations": kurt, "ic": {f"{a}|{b}": v for (a, b), v in ic_tab.stack(level=1, future_stack=True)["mean_IC"].items()},
           "ic_t": {f"{a}|{b}": v for (a, b), v in ic_tab.stack(level=1, future_stack=True)["t_NW"].items()},
           "bucket_bps": {f"{a}|{h}": {str(k): float(x) for k, x in s.items()} for (a, h), s in bucket_tab.items()},
           "regimes": reg_tab.to_dict(orient="index"), "readings": readings,
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "s08": s08.S08_VERSION, "panel": s04.PANEL_VERSION}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
exc.to_csv(OUT / "exceedance.csv"); by_slot.to_csv(OUT / "exceedance_by_slot.csv"); ic_tab.to_csv(OUT / "ic.csv")
bucket_tab.to_csv(OUT / "bucket_contrarian_bps.csv"); reg_tab.to_csv(OUT / "regimes.csv")
print(json.dumps({k: bool(v) for k, v in readings.items()}, indent=2))